# Module 1 — OOP · Practice Solutions
> **Teacher's solution guide.** Each question shows the **concept** it tests, a short **how-to-explain** note, the **worked solution** (run it live), and a **common mistake** to highlight. Run every code cell with `Shift+Enter`.

### Q1. `Rectangle` with `width`, `height`, `area()` and `__str__`
**Concept:** a class bundles per-object *data* (`__init__`) with *behaviour* (methods); `__str__` is the human-readable form.

**Explain to students:** `self` is *this* rectangle; `area()` uses its own data; `print(r)` automatically calls `__str__`.

**Common mistake:** forgetting `self.` (writing `width` instead of `self.width`).

In [1]:
class Rectangle:
    def __init__(self, width, height):
        self.width = width        # per-object data
        self.height = height
    def area(self):               # behaviour
        return self.width * self.height
    def __str__(self):            # used by print()/str()
        return f'Rectangle({self.width} x {self.height}), area = {self.area()}'

r = Rectangle(4, 3)
print(r)
print('Area:', r.area())

Rectangle(4 x 3), area = 12
Area: 12


### Q2. `Square` subclass reusing `Rectangle` via `super()`
**Concept:** inheritance + the *is-a* relationship; `super().__init__` reuses the parent constructor instead of copying code.

**Explain to students:** a square is a rectangle with equal sides; it inherits `area()` and `__str__` for free.

**Common mistake:** re-writing width/height logic instead of calling `super()`.

In [2]:
class Square(Rectangle):
    def __init__(self, side):
        super().__init__(side, side)   # reuse parent init

sq = Square(5)
print(sq)              # inherited __str__
print('Area:', sq.area())   # inherited area()

Rectangle(5 x 5), area = 25
Area: 25


### Q3. `BankAccount` with a private balance and validated deposit/withdraw
**Concept:** encapsulation — hide state (`__balance`), expose a safe interface that rejects invalid operations.

**Explain to students:** the double underscore makes `balance` private (name-mangled), so no one can set a negative balance directly; all changes go through methods that validate.

**Common mistake:** making `balance` a public attribute that any code can corrupt.

In [3]:
class BankAccount:
    def __init__(self, owner, balance=0):
        self.owner = owner
        self.__balance = balance          # private
    @property
    def balance(self):                    # read-only view
        return self.__balance
    def deposit(self, amount):
        if amount <= 0:
            raise ValueError('deposit must be positive')
        self.__balance += amount
    def withdraw(self, amount):
        if amount <= 0:
            raise ValueError('withdrawal must be positive')
        if amount > self.__balance:
            raise ValueError('insufficient funds')
        self.__balance -= amount

acc = BankAccount('Asha', 1000)
acc.deposit(500); acc.withdraw(200)
print('Balance:', acc.balance)
for action, amt in [(acc.deposit, -50), (acc.withdraw, 99999)]:
    try:
        action(amt)
    except ValueError as e:
        print('Rejected:', e)

Balance: 1300
Rejected: deposit must be positive
Rejected: insufficient funds


### Q4. Abstract `Shape` with `area()`; implement `Circle` and `Triangle`
**Concept:** abstraction — define *what* every shape must do (a contract) without saying *how*; subclasses fill it in.

**Explain to students:** `Shape` can't be created on its own; it forces each shape to provide `area()`. The loop then treats them uniformly — that's polymorphism.

**Common mistake:** trying to instantiate the abstract class, or forgetting `@abstractmethod`.

In [4]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    @abstractmethod
    def area(self):
        ...

class Circle(Shape):
    def __init__(self, r): self.r = r
    def area(self): return math.pi * self.r ** 2

class Triangle(Shape):
    def __init__(self, base, height): self.base, self.height = base, height
    def area(self): return 0.5 * self.base * self.height

for sh in [Circle(2), Triangle(3, 4)]:
    print(type(sh).__name__, '->', round(sh.area(), 2))

try:
    Shape()
except TypeError as e:
    print('Cannot instantiate Shape:', e)

Circle -> 12.57
Triangle -> 6.0
Cannot instantiate Shape: Can't instantiate abstract class Shape without an implementation for abstract method 'area'


### Q5. Overload `__add__` on a `Vector2D` so `v1 + v2` works
**Concept:** operator overloading (a form of polymorphism) via dunder methods.

**Explain to students:** Python turns `v1 + v2` into `v1.__add__(v2)`. We return a **new** vector and add `__repr__` so it prints clearly.

**Common mistake:** mutating `self` instead of returning a new object.

In [5]:
class Vector2D:
    def __init__(self, x, y): self.x, self.y = x, y
    def __add__(self, other):
        return Vector2D(self.x + other.x, self.y + other.y)
    def __repr__(self):
        return f'Vector2D({self.x}, {self.y})'

print(Vector2D(1, 2) + Vector2D(3, 4))   # Vector2D(4, 6)

Vector2D(4, 6)


### Q6. `Student.from_csv_row('Ravi,8.4')` — an alternative constructor
**Concept:** `@classmethod` receives the class as `cls` and can build and return an instance — a *factory* / alternative constructor.

**Explain to students:** the normal constructor takes ready values; this one takes a raw CSV line, parses it, and builds the object. Using `cls(...)` (not `Student(...)`) means subclasses get the correct type.

**Common mistake:** forgetting to convert `cgpa` to `float` (it arrives as text).

In [6]:
class Student:
    def __init__(self, name, cgpa):
        self.name = name
        self.cgpa = cgpa
    @classmethod
    def from_csv_row(cls, row):
        name, cgpa = row.split(',')
        return cls(name, float(cgpa))
    def __repr__(self):
        return f'Student({self.name!r}, {self.cgpa})'

s = Student.from_csv_row('Ravi,8.4')
print(s)

Student('Ravi', 8.4)
